# Assignment 3 – Agentic RAG Pipeline

This notebook builds an agentic Retrieval-Augmented Generation (RAG) system that answers questions about five landmark
language-model papers: the Transformer (2017), BERT (2018), GPT-3 (2020), LLaMA (2023) and DeepSeek-R1 (2025).
The PDFs are downloaded from arXiv, their text is extracted with `pypdf`, split into chunks, embedded and stored in a
FAISS index, and every chunk carries `source` and `date` metadata.

For every question, an LLM agent first decides whether the documents need to be searched, rewrites the search query and
chooses metadata filters. The search then runs only over the chunks that pass those filters, and the LLM answers from the
retrieved passages with citations, or says "I don't know" when the documents don't cover the question.

The **Models** used for this assignment are: embeddings `all-mpnet-base-v2` (replacing `all-MiniLM-L6-v2`), re-ranker `cross-encoder/ms-marco-MiniLM-L-6-v2`,
LLM `Qwen/Qwen3-4B`.

**Requires a `GPU runtime`. On `CPU` set `LLM_MODEL = "Qwen/Qwen3-1.7B"`.**

## Required installs & Import Statements
Installs the libraries the pipeline needs: `sentence-transformers` (the embedding model and the re-ranker), `faiss-cpu` (the vector index that finds the nearest vectors to a question), `pypdf` (extracts text from the PDFs) and `rank-bm25` (keyword search). `torch`, `transformers` and `requests` are already installed on Colab. The models themselves are downloaded later, the first time each one is loaded.

In [1]:
!pip install -q sentence-transformers faiss-cpu pypdf rank-bm25

In [2]:
import os
import re
import json
import requests
from pypdf import PdfReader
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on:", DEVICE)

Running on: cuda


## Knowledge base - loading the documents

Download the five pdf documents from arXiv, extracts the text of the first `MAX_PAGES` of each document and cleans it.

Every document is then structured with the title, source, and date which become the metadata for the filtering.

In [3]:
DOCS_DIR = "documents"
MAX_PAGES = 8
os.makedirs(DOCS_DIR, exist_ok=True)

KNOWLEDGE_BASE = [
    {"file": "vaswani_2017_attention.pdf", "url": "https://arxiv.org/pdf/1706.03762",
        "title": "Attention Is All You Need", "source": "Google Brain", "date": "2017-06-12"},
    {"file": "devlin_2018_bert.pdf", "url": "https://arxiv.org/pdf/1810.04805",
        "title": "BERT: Pre-training of Deep Bidirectional Transformers", "source": "Google AI", "date": "2018-10-11"},
    {"file": "brown_2020_gpt3.pdf", "url": "https://arxiv.org/pdf/2005.14165",
        "title": "Language Models are Few-Shot Learners (GPT-3)", "source": "OpenAI", "date": "2020-05-28"},
    {"file": "touvron_2023_llama.pdf", "url": "https://arxiv.org/pdf/2302.13971",
        "title": "LLaMA: Open and Efficient Foundation Language Models", "source": "Meta AI", "date": "2023-02-27"},
    {"file": "deepseek_2025_r1.pdf", "url": "https://arxiv.org/pdf/2501.12948",
        "title": "DeepSeek-R1: Incentivizing Reasoning Capability in LLMs via RL", "source": "DeepSeek", "date": "2025-01-22"},
]

def download_pdf(doc):
    path = os.path.join(DOCS_DIR, doc["file"])
    if not os.path.exists(path):
        response = requests.get(doc["url"], timeout=60)
        response.raise_for_status()
        with open(path, "wb") as f:
            f.write(response.content)
    return path

def extract_text(pdf_path, max_pages=None):
    reader = PdfReader(pdf_path)
    pages = reader.pages[:max_pages] if max_pages else reader.pages
    text = " ".join(page.extract_text() or "" for page in pages)
    text = " ".join(text.split())
    return re.sub(r"(\w)- (\w)", r"\1\2", text)

raw_documents = []
for doc in KNOWLEDGE_BASE:
    path = download_pdf(doc)
    text = extract_text(path, MAX_PAGES)
    raw_documents.append({**doc, "text": text})
    print(f"Loaded   {doc['file']:<30} {len(text.split()):>6} words")

Loaded   vaswani_2017_attention.pdf       3818 words
Loaded   devlin_2018_bert.pdf             5207 words
Loaded   brown_2020_gpt3.pdf              4872 words
Loaded   touvron_2023_llama.pdf           4660 words
Loaded   deepseek_2025_r1.pdf             3490 words


## Chunking and metadata

Split the document texts into overlapping word based chunks (150 words) and create a full dictionary with the title, source, date and the chunk lists to form the metadata store. Chunks that are mostly numbers are dropped as they carry no readable information.

In [4]:
CHUNK_SIZE = 150
OVERLAP = 30

def chunk_words(text, chunk_size=CHUNK_SIZE, overlap=OVERLAP):
    words = text.split()
    if not words:
        return []
    step = chunk_size - overlap
    last_start = max(len(words) - overlap, 1)
    return [" ".join(words[i:i + chunk_size]) for i in range(0, last_start, step)]

def is_mostly_text(chunk_text, min_ratio=0.5):
    words = chunk_text.split()
    real_words = sum(re.sub(r"\W", "", w).isalpha() for w in words)
    return real_words / len(words) >= min_ratio

chunks = []
for doc in raw_documents:
    pieces = [p for p in chunk_words(doc["text"]) if is_mostly_text(p)]
    for n, piece in enumerate(pieces):
        chunks.append({
            "chunk_id": f"{doc['file']}#{n}",
            "title": doc["title"],
            "source": doc["source"],
            "date": doc["date"],
            "text": piece,
        })

print(f"{len(chunks)} chunks from {len(raw_documents)} documents\n")
print({k: v for k, v in chunks[0].items() if k != "text"})
print(chunks[0]["text"][:300], "...")

177 chunks from 5 documents

{'chunk_id': 'vaswani_2017_attention.pdf#0', 'title': 'Attention Is All You Need', 'source': 'Google Brain', 'date': '2017-06-12'}
Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works. Attention Is All You Need Ashish Vaswani∗ Google Brain avaswani@google.com Noam Shazeer∗ Google Brain noam@google.com Niki Par ...


## Embeddings and FAISS Indexing

Turn each chunk into a 768-number vector using the `all-mpnet-base-v2` embedding model which is slower but a lot more accurate than the `all-MiniLM-L6-v2` model. The vectors are normalized to length 1 so the inner product that `IndexFlatIP` computes is exactly the cosine similarity.

In [5]:
EMBEDDING_MODEL = "sentence-transformers/all-mpnet-base-v2"
embedder = SentenceTransformer(EMBEDDING_MODEL, device=DEVICE)

embeddings = embedder.encode([c["text"] for c in chunks], show_progress_bar=True, convert_to_numpy=True)
embeddings = embeddings.astype("float32")
faiss.normalize_L2(embeddings)

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

print(f"Embedding dimension: {embeddings.shape[1]}")
print(f"Vectors in the FAISS index: {index.ntotal}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Embedding dimension: 768
Vectors in the FAISS index: 177


### Keyword index (BM25) and re-ranker

Define the `BM25` index to rank chunks by exact word overlap with the query (sparse retrieval), and add a `cross-encoder` re-ranker that reads the question and a chunk together and gives a relevance score.

The re-ranker only runs on the reordred candidate chunks the FAISS and BM25 found.

In [6]:
def tokenize(text):
    return re.findall(r"\w+", text.lower())

bm25 = BM25Okapi([tokenize(c["text"]) for c in chunks])
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", device=DEVICE)
print("BM25 index and re-ranker ready.")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BM25 index and re-ranker ready.


## Retrieval with metadata filtering

`retrieve()` works in three steps:

1. **Metadata filter first:** `matches_filters()` keeps only the ids of chunks whose `source` and year match the filters.
2. **Candidate search, only among those ids:** `faiss_search()` embeds the query with the *same* model used for the
   chunks and passes the allowed ids to FAISS with an `IDSelectorBatch`, so FAISS ranks **only** the chunks that passed
   the filter. `keyword_search()` ranks the same allowed ids with BM25. Both run for the user's question and for the
   agent's rewritten search query.
3. **Re-ranking:** the cross-encoder scores every candidate against the question, and the best `top_k` are returned.

In [7]:
def matches_filters(chunk, source=None, year_from=None, year_to=None):
    year = int(chunk["date"][:4])
    if source and source.lower() not in chunk["source"].lower():
        return False
    if year_from and year < year_from:
        return False
    if year_to and year > year_to:
        return False
    return True

def faiss_search(query, allowed_ids, n):
    query_vec = embedder.encode([query], convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(query_vec)
    selector = faiss.IDSelectorBatch(len(allowed_ids), faiss.swig_ptr(allowed_ids))
    _, ids = index.search(query_vec, n, params=faiss.SearchParameters(sel=selector))
    return [int(i) for i in ids[0] if i != -1]

def keyword_search(query, allowed_ids, n):
    scores = bm25.get_scores(tokenize(query))
    ranked = sorted(allowed_ids, key=lambda i: scores[i], reverse=True)
    return [int(i) for i in ranked[:n] if scores[i] > 0]

def retrieve(question, search_query=None, top_k=3, source=None, year_from=None, year_to=None, n_candidates=20):
    allowed_ids = np.array(
        [i for i, c in enumerate(chunks) if matches_filters(c, source, year_from, year_to)],
        dtype="int64",
    )
    if len(allowed_ids) == 0:
        return []

    queries = [question] if not search_query or search_query == question else [question, search_query]
    candidates = []
    for q in queries:
        candidates += faiss_search(q, allowed_ids, n_candidates)
        candidates += keyword_search(q, allowed_ids, n_candidates)
    candidates = list(dict.fromkeys(candidates))

    scores = reranker.predict([(question, chunks[i]["text"]) for i in candidates])
    best = np.argsort(-scores)[:top_k]
    return [{**chunks[candidates[j]], "score": float(scores[j])} for j in best]

## Retrieval Tests

Compare FAISS with the full `retrieve()` function.

First, FAISS on its own versus the full `retrieve()`: dense search alone misses the chunk that names `SwiGLU`, while
hybrid search plus re-ranking puts it first. Then the same kind of question is searched with different `source` and
year filters, to show that the metadata changes which chunks can be returned. The last filter combination matches no
document, so nothing is returned.

In [8]:
def show(results):
    for r in results:
        print(f"  {r['score']:6.2f} | {r['source']:<12} | {r['date']} | {r['text'][:70]}...")
    if not results:
        print("  (no chunk passed the filter)")

question = "What activation function does LLaMA use instead of ReLU?"
all_ids = np.arange(len(chunks), dtype="int64")

print("FAISS only, top 3 (does the chunk mention SwiGLU?):")
for i in faiss_search(question, all_ids, 3):
    print(f"  {chunks[i]['chunk_id']:<32} {'SwiGLU' in chunks[i]['text']}")

print("\nFull retrieve(), no filter:")
show(retrieve(question))

print("\nsource = 'Meta AI':")
show(retrieve(question, source="Meta AI"))

print("\nyear_from = 2024:")
show(retrieve("How is reinforcement learning used to train the model?", year_from=2024))

print("\nsource = 'Google', year_to = 2017:")
show(retrieve("What is the training objective?", source="Google", year_to=2017))

print("\nsource = 'DeepSeek', year_to = 2018 (impossible combination):")
show(retrieve("What is the training objective?", source="DeepSeek", year_to=2018))

FAISS only, top 3 (does the chunk mention SwiGLU?):
  touvron_2023_llama.pdf#27        False
  touvron_2023_llama.pdf#32        False
  touvron_2023_llama.pdf#31        False

Full retrieve(), no filter:
    2.63 | Meta AI      | 2023-02-27 | the main difference with the original architecture, and where we were ...
   -3.28 | Google Brain | 2017-06-12 | layers in our encoder and decoder contains a fully connected feed-forw...
   -4.69 | Meta AI      | 2023-02-27 | able to follow basic instructions, we observe that a very small amount...

source = 'Meta AI':
    2.63 | Meta AI      | 2023-02-27 | the main difference with the original architecture, and where we were ...
   -4.69 | Meta AI      | 2023-02-27 | able to follow basic instructions, we observe that a very small amount...
   -4.87 | Meta AI      | 2023-02-27 | of our instruct model LLaMA-I on MMLU and compare with existing instru...

year_from = 2024:
    2.68 | DeepSeek     | 2025-01-22 | for understanding the mechanisms underl

### FAISS search with a metadata filter

The same question searched by FAISS alone: first over all chunks, then only over the chunks whose `source` is OpenAI.
The filter is passed into the FAISS search itself (`IDSelectorBatch`), so the metadata is applied **before** FAISS ranks.

In [9]:
arch_question = "Which model architecture is used?"
openai_ids = np.array([i for i, c in enumerate(chunks) if matches_filters(c, source="OpenAI")], dtype="int64")

for label, ids in [("no filter", all_ids), ("source = 'OpenAI'", openai_ids)]:
    hits = faiss_search(arch_question, ids, 3)
    print(f"{label:<18}", [f"{chunks[i]['source']} ({chunks[i]['date'][:4]})" for i in hits])

no filter          ['Meta AI (2023)', 'Google Brain (2017)', 'DeepSeek (2025)']
source = 'OpenAI'  ['OpenAI (2020)', 'OpenAI (2020)', 'OpenAI (2020)']


## LLM - Hugging Face Model

Instead of the template's default `google/flan-t5-base` (an encoder-decoder model), this assignment uses **`Qwen/Qwen3-4B`**, a larger decoder-only, instruction-tuned model from Alibaba (S6, slide 11). It follows instructions well enough to act as the agent and reply in JSON, and it answers more accurately than the 1.7B version I tried first.

Qwen3 has a "thinking" and a "non-thinking" mode. Thinking is turned off (`enable_thinking=False`) because answering from retrieved passages doesn't need long reasoning, and it's much faster. `do_sample=False` makes the output deterministic.

`generate()` takes a list of chat messages, applies the model's chat template, and returns only the newly generated text.

In [10]:
LLM_MODEL = "Qwen/Qwen3-4B"

tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)
llm = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL,
    dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
).to(DEVICE)

def generate(messages, max_new_tokens=256):
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        output = llm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    text = tokenizer.decode(new_tokens, skip_special_tokens=True)
    return re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()

print(generate([{"role": "user", "content": "In one sentence, what is retrieval-augmented generation?"}]))

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Retrieval-augmented generation is a technique that combines the ability to retrieve relevant information from a database or corpus with the generation of coherent, context-aware text.


## Define the Agent

The agent is the LLM itself, given a router prompt that describes the search tool and the document catalogue, with a few
examples of the expected JSON reply (few-shot prompting). From that, it decides **whether to retrieve**, **with what
search query**, and **which filters** (`source`, `year_from`, `year_to`) to apply. The reply is parsed and validated: a
source that doesn't exist is dropped and years must be numbers. If the reply isn't valid JSON, a rule-based fallback
decides instead, and `decided_by` shows which one made the decision.

In [11]:
KNOWN_SOURCES = sorted({c["source"] for c in chunks})
CATALOG = "\n".join(f"- {d['title']} ({d['source']}, {d['date']})" for d in raw_documents)

ROUTER_PROMPT = f"""You are the routing agent of a question-answering system.
The system has a search tool over this document collection:
{CATALOG}

Decide whether the user's message needs the search tool.
- Use it for EVERY question that asks for information, even if the topic seems unrelated to the collection
  (the system will report when the documents don't cover it). Never answer factual questions from memory.
- Do not use it only for greetings, thanks, small talk or questions about yourself.
- When in doubt, set "retrieve" to true.
- Only set "source", "year_from" or "year_to" when the user's message explicitly names an organisation or a year;
  never infer them from the catalogue.

Reply with ONLY a JSON object with these keys:
  "retrieve": true or false
  "search_query": a short keyword query for the search tool: keep the technical terms, drop filler words and
                  leave out the organisation and the years (those go in the filters), or "" if retrieve is false
  "source": one of {KNOWN_SOURCES} if the user limits the question to one organisation, otherwise null
  "year_from": the earliest publication year the user asks for, otherwise null
  "year_to": the latest publication year the user asks for, otherwise null

Examples:
User: Hi there!
{{"retrieve": false, "search_query": "", "source": null, "year_from": null, "year_to": null}}
User: What did Google Brain propose in 2017 to replace recurrence?
{{"retrieve": true, "search_query": "replace recurrence attention mechanism", "source": "Google Brain", "year_from": 2017, "year_to": 2017}}
User: Which papers after 2022 talk about training only on public data?
{{"retrieve": true, "search_query": "training on publicly available datasets", "source": null, "year_from": 2023, "year_to": null}}
User: Who won the football World Cup in 2010?
{{"retrieve": true, "search_query": "football World Cup winner", "source": null, "year_from": null, "year_to": null}}"""


def as_year(value):
    try:
        return int(value)
    except (TypeError, ValueError):
        return None

def parse_decision(text):
    """Turn the LLM's reply into a clean decision dict, or None if it isn't valid JSON."""
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if not match:
        return None
    try:
        raw = json.loads(match.group(0))
    except json.JSONDecodeError:
        return None
    source = raw.get("source")
    return {
        "retrieve": raw.get("retrieve") in (True, "true", "True"),
        "search_query": str(raw.get("search_query") or "").strip(),
        "source": source if source in KNOWN_SOURCES else None,
        "year_from": as_year(raw.get("year_from")),
        "year_to": as_year(raw.get("year_to")),
    }

SMALL_TALK = re.compile(r"^\s*(hi|hello|hey|thanks|thank you|good (morning|afternoon|evening))\b", re.IGNORECASE)

def rule_based_decision(user_input):
    """Fallback router: keyword rules, used only when the LLM's reply can't be parsed."""
    years = [int(y) for y in re.findall(r"\b(?:19|20)\d{2}\b", user_input)]
    return {
        "retrieve": not (SMALL_TALK.match(user_input) and len(user_input.split()) <= 6),
        "search_query": user_input,
        "source": next((s for s in KNOWN_SOURCES if s.lower() in user_input.lower()), None),
        "year_from": min(years) if years else None,
        "year_to": max(years) if years else None,
    }

def agent_decide(user_input):
    reply = generate(
        [{"role": "system", "content": ROUTER_PROMPT}, {"role": "user", "content": user_input}],
        max_new_tokens=100,
    )
    decision = parse_decision(reply)
    if decision is None:
        decision = rule_based_decision(user_input)
        decision["decided_by"] = "rules (fallback)"
    else:
        decision["decided_by"] = "LLM"
    if decision["retrieve"] and not decision["search_query"]:
        decision["search_query"] = user_input
    return decision

print(agent_decide("Hello!"))
print(agent_decide("What did OpenAI show about few-shot learning in 2020?"))

{'retrieve': False, 'search_query': '', 'source': None, 'year_from': None, 'year_to': None, 'decided_by': 'LLM'}
{'retrieve': True, 'search_query': 'few-shot learning', 'source': 'OpenAI', 'year_from': 2020, 'year_to': 2020, 'decided_by': 'LLM'}


## Full agentic RAG pipeline

`run_agentic_rag()` connects all the pieces:

1. The **agent** decides whether to retrieve, the search query and the filters.
2. No retrieval → the LLM replies directly, without stating any facts.
3. Retrieval → metadata filter, then FAISS + BM25 search and re-ranking, for the top 4 chunks.
4. If even the best chunk's re-ranker score is below `RELEVANCE_THRESHOLD`, the documents don't cover the question, so
   the pipeline answers "I don't know" instead of hallucinating.
5. Otherwise the chunks are numbered and added to the prompt (augmentation), with instructions to use **only** that
   context, cite the passages like `[1]`, and say "I don't know" when the answer isn't there.
6. The answer is returned together with its sources.

In [12]:
RELEVANCE_THRESHOLD = -3.0
NO_ANSWER = "I don't know based on the provided documents."

ANSWER_PROMPT = """Answer the question using ONLY the numbered context passages below.
Cite the passages you used, like [1] or [2]. Keep the answer short.
If the context does not contain the answer, reply exactly: "I don't know based on the provided documents."

Context:
{context}"""

def answer_with_context(question, passages):
    context = "\n\n".join(
        f"[{n}] ({p['title']}, {p['source']}, {p['date']})\n{p['text']}"
        for n, p in enumerate(passages, start=1)
    )
    return generate([
        {"role": "system", "content": ANSWER_PROMPT.format(context=context)},
        {"role": "user", "content": question},
    ])


def run_agentic_rag(user_input, top_k=4):
    print("=" * 90)
    print("QUESTION:", user_input)

    decision = agent_decide(user_input)
    print(f"AGENT ({decision['decided_by']}): retrieve={decision['retrieve']}, "
            f"query={decision['search_query']!r}, source={decision['source']}, "
            f"years={decision['year_from']}-{decision['year_to']}")

    passages = []
    if not decision["retrieve"]:
        answer = generate([
            {"role": "system", "content": "You are a friendly assistant for a collection of language-model "
                                            "research papers. Reply briefly. Do not state any facts: if the user "
                                            "asks for information, invite them to ask about the papers."},
            {"role": "user", "content": user_input},
        ], max_new_tokens=80)
    else:
        passages = retrieve(user_input, decision["search_query"], top_k, decision["source"],
                            decision["year_from"], decision["year_to"])
        best = passages[0]["score"] if passages else float("-inf")
        print(f"RETRIEVED: {len(passages)} chunks, best re-ranker score {best:.2f}")
        if best < RELEVANCE_THRESHOLD:
            answer = NO_ANSWER
        else:
            answer = answer_with_context(user_input, passages)

    print("\nANSWER:", answer)
    if passages and answer != NO_ANSWER:
        print("\nSOURCES:")
        for n, p in enumerate(passages, start=1):
            print(f"  [{n}] {p['title']} – {p['source']}, {p['date']} (score {p['score']:.2f})")
    print()
    return {"question": user_input, "decision": decision, "passages": passages, "answer": answer}

## Pipeline tests

Define different type of questions to test the solidity of the Agentic RAG.

Each question tests a different path through the pipeline:

1. A normal question → retrieval and a cited answer.
2. A question naming an organisation and a year → `source` and date filters.
3. A question with a date range → year-range filter.
4. Filters that no document matches (DeepSeek didn't publish in 2018) → no chunks, so "I don't know".
5. A question the documents can't answer → the re-ranker score falls below the threshold → "I don't know".
6. An out-of-scope product question → no made-up answer.
7. A greeting → the agent skips retrieval.

In [13]:
test_questions = [
    "What activation function does LLaMA use instead of ReLU?",
    "What pre-training tasks does BERT use, according to Google AI's 2018 paper?",
    "In papers published after 2024, how is reinforcement learning used to improve reasoning?",
    "What did DeepSeek publish in 2018 about attention?",
    "According to these papers, what is the capital of Australia?",
    "What is the battery capacity of the iPhone 18 Pro?",
    "Hello! Can you help me today?",
]

results = [run_agentic_rag(q) for q in test_questions]

QUESTION: What activation function does LLaMA use instead of ReLU?
AGENT (LLM): retrieve=True, query='activation function used by llama', source=Meta AI, years=2023-2023
RETRIEVED: 4 chunks, best re-ranker score 2.63

ANSWER: The activation function used by LLaMA instead of ReLU is SwiGLU, as mentioned in [1].

SOURCES:
  [1] LLaMA: Open and Efficient Foundation Language Models – Meta AI, 2023-02-27 (score 2.63)
  [2] LLaMA: Open and Efficient Foundation Language Models – Meta AI, 2023-02-27 (score -4.69)
  [3] LLaMA: Open and Efficient Foundation Language Models – Meta AI, 2023-02-27 (score -4.87)
  [4] LLaMA: Open and Efficient Foundation Language Models – Meta AI, 2023-02-27 (score -5.20)

QUESTION: What pre-training tasks does BERT use, according to Google AI's 2018 paper?
AGENT (LLM): retrieve=True, query='pre-training tasks BERT', source=Google AI, years=2018-2018
RETRIEVED: 4 chunks, best re-ranker score 6.36

ANSWER: BERT uses two unsupervised pre-training tasks: Masked LM and 

## Evalutaion

### Faithfulness check - Semantic Similarity

Every sentence of an answer is embedded and compared with the retrieved chunks. A sentence that isn't similar to any chunk is probably not supported by the context. The Hallucination score is the difference between 1 - best cosine similarity where sentences above 0.7 are flagged.

In [14]:
def faithfulness_check(result, flag_above=0.7):
    if not result["passages"] or result["answer"] == NO_ANSWER:
        return
    answer = re.sub(r"\s*\[\d+\]", "", result["answer"])
    sentences = [s for s in re.split(r"(?<=[.!?])\s+", answer) if len(s.split()) >= 3]
    if not sentences:
        return
    sent_vecs = embedder.encode(sentences, convert_to_numpy=True).astype("float32")
    ctx_vecs = embedder.encode([p["text"] for p in result["passages"]], convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(sent_vecs)
    faiss.normalize_L2(ctx_vecs)
    best_similarity = (sent_vecs @ ctx_vecs.T).max(axis=1)

    print("QUESTION:", result["question"])
    for sentence, sim in zip(sentences, best_similarity):
        score = 1 - sim
        flag = "  <-- possibly unsupported" if score > flag_above else ""
        print(f"  {score:.2f}  {sentence[:90]}{flag}")
    print()

for r in results:
    faithfulness_check(r)

QUESTION: What activation function does LLaMA use instead of ReLU?
  0.57  The activation function used by LLaMA instead of ReLU is SwiGLU, as mentioned in.

QUESTION: What pre-training tasks does BERT use, according to Google AI's 2018 paper?
  0.23  BERT uses two unsupervised pre-training tasks: Masked LM and Next Sentence Prediction.
  0.71  These tasks are described in the paper.  <-- possibly unsupported

QUESTION: In papers published after 2024, how is reinforcement learning used to improve reasoning?
  0.26  Reinforcement learning is used to improve reasoning by incentivizing the emergence of adva
  0.36  The approach employs Group Relative Policy Optimization (GRPO) and focuses on the correctn
  0.48  This method allows for the development of diverse and sophisticated reasoning capabilities



## RAGAs

RAGAs uses an OpenAI model as the judge, so it needs an `OPENAI_API_KEY` with credits. It is switched off
(`RUN_RAGAS = False`) so the notebook runs without errors: my OpenAI account has no credits, and the installed `ragas`
version currently fails to import because of a `langchain-community` version conflict. The faithfulness check above
covers the same idea without an external API.


In [15]:
RUN_RAGAS = False

if RUN_RAGAS:
    !pip install -q ragas datasets
    from google.colab import userdata
    from datasets import Dataset
    from ragas import evaluate
    from ragas.metrics import faithfulness, answer_relevancy

    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    rows = [r for r in results if r["passages"] and r["answer"] != NO_ANSWER]
    eval_dataset = Dataset.from_dict({
        "question": [r["question"] for r in rows],
        "answer": [r["answer"] for r in rows],
        "contexts": [[p["text"] for p in r["passages"]] for r in rows],
    })
    print(evaluate(eval_dataset, metrics=[faithfulness, answer_relevancy]).to_pandas())
else:
    print("RAGAs skipped (RUN_RAGAS = False).")

RAGAs skipped (RUN_RAGAS = False).
